# Slice aligner: crossing vs non-crossing trees (t = 7 -> 8)

Load lineage subtrees spanning **t=7 -> t=8** from the crossing and non-crossing
hierarchical-OU sims, load the **slice aligner** (`slice_marginal_ca_path`:
reference-bank time-marginal + root-to-leaf path + common-ancestor triples), and
compute the energy distance between sets of trees. `last_feature_losses` gives the
per-feature breakdown (`time`=marginal, `slice`=path, `common_ancestor`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import hydra
import networkx as nx
import numpy as np
import torch
from omegaconf import open_dict

from datasets.tree_record import collate_subtrees
from synthetic_evals.core.hydra_io import DEVICE, compose_cfg
from synthetic_evals.core.crossing_eval import build_eval_sim
from synthetic_evals.core.subtrees import build_layer_subtrees

T_SRC, T_TGT = 7.0, 8.0
EXP = "high_dim_crossing/unshared_vpsde_idnn_leaf"   # carries the slice aligner
PROJ = False                                          # False -> native 1-D
print("device:", DEVICE)

In [ ]:
def make_sim(variant, seed):
    """Held-out crossing/non-crossing sim. PROJ=False strips the projector (1-D)."""
    ds = compose_cfg(EXP, f"high_dim_crossing/hierarchical_ou_{variant}").dataset
    if not PROJ:
        with open_dict(ds):
            ds.pop("proj_dim", None)
    return build_eval_sim(ds, seed=seed)


def _relabel(subs, tag):
    """Prefix node names so subtrees from different sims stay unique on collate."""
    out = []
    for s in subs:
        m = {n: f"{tag}_{n}" for n in s["graph"].nodes}
        out.append({**s,
                    "graph": nx.relabel_nodes(s["graph"], m),
                    "root": m[s["root"]],
                    "leaves": [(m[ln], orig) for ln, orig in s["leaves"]],
                    "leaf_states": {m[ln]: v for ln, v in s["leaf_states"].items()}})
    return out


def collate_states(subs):
    """(batch, states) from a list of subtree dicts; true root@7 + leaf@8 states."""
    sdicts = [{"graph": s["graph"], "root": s["root"],
               "leaves": [n for n, _ in s["leaves"]]} for s in subs]
    leaf_states = {n: s["leaf_states"][n] for s in subs for n, _ in s["leaves"]}
    batch = collate_subtrees(sdicts, leaf_states, device=DEVICE, randomize_child_index=False)
    B, N = batch.mask.shape
    dim = next(iter(leaf_states.values())).shape[-1]
    states = torch.zeros(B, N, dim, device=DEVICE)
    for i, s in enumerate(subs):
        idx = {nm: j for j, nm in enumerate(batch.node_names[i])}
        states[i, batch.root_idx[i]] = torch.as_tensor(s["x_src"], dtype=torch.float32, device=DEVICE)
        for ln, _ in s["leaves"]:
            states[i, idx[ln]] = s["leaf_states"][ln].to(DEVICE)
    return batch, states

In [ ]:
# Load the slice aligner and bind a dataset (reference bank for the marginal feature).
cfg = compose_cfg(EXP, "high_dim_crossing/hierarchical_ou_crossing")
aligner = hydra.utils.instantiate(cfg.aligner).to(DEVICE)
if not PROJ:
    with open_dict(cfg.dataset):
        cfg.dataset.pop("proj_dim", None)
aligner.bind_dataset(hydra.utils.instantiate(cfg.dataset))

print("aligner:", type(aligner).__name__)
for f, w in zip(aligner.features, aligner.weights):
    nm = getattr(f, "name", type(f).__name__)
    print(f"  weight={w:<5} {type(f).__name__:24s} -> '{nm}'")

In [ ]:
# Single-pair sanity check: all trees, crossing vs non-crossing.
def build_set(variant, seeds):
    subs = []
    for seed in seeds:
        subs += _relabel(build_layer_subtrees(make_sim(variant, seed), T_SRC, T_TGT), f"{variant}{seed}")
    b, s = collate_states(subs)
    return b, s, len(subs)

bc, sc, nc = build_set("crossing", range(4))
bnc, snc, nnc = build_set("noncrossing", range(4))
print(f"crossing trees: {nc}   non-crossing trees: {nnc}\n")
with torch.no_grad():
    align = aligner.forward(bc, sc, bnc, snc)
print(f"ALIGNMENT (crossing vs non-crossing, t=7->8): {float(align):.5f}\n")
for k, v in aligner.last_feature_losses.items():
    print(f"  {k:18s} {v:.5f}")

## Pairwise grid of crossing/coherent mixtures

Mixtures with a varying fraction of crossing trees (0%->100%); energy distance between
**every pair**, the **mean over `R` fresh mixture draws** to suppress noise. The same
pass also records each feature's contribution (`time`/`slice`/`common_ancestor`) via
`aligner.last_feature_losses`, so the same pass gives per-feature grids.

In [ ]:
POOL_SEEDS = list(range(12))      # ~768 trees per regime
FRACTIONS  = np.linspace(0.0, 1.0, 6)
N_MIX      = 256                  # trees per mixture
R          = 6                    # fresh mixture draws averaged per grid cell
RNG        = np.random.default_rng(0)

cross_pool = []
noncross_pool = []
for seed in POOL_SEEDS:
    cross_pool    += _relabel(build_layer_subtrees(make_sim("crossing", seed),    T_SRC, T_TGT), f"c{seed}")
    noncross_pool += _relabel(build_layer_subtrees(make_sim("noncrossing", seed), T_SRC, T_TGT), f"n{seed}")
print(f"pools: crossing={len(cross_pool)}  noncrossing={len(noncross_pool)}")

def mixture(frac):
    n_c = int(round(frac * N_MIX))
    c  = [cross_pool[i]    for i in RNG.choice(len(cross_pool),    n_c,         replace=False)]
    nc = [noncross_pool[i] for i in RNG.choice(len(noncross_pool), N_MIX - n_c, replace=False)]
    return collate_states(c + nc)

In [ ]:
# One pass: total energy distance D plus each feature's grid G[k].
FEAT_KEYS = ["time", "slice", "common_ancestor"]
K = len(FRACTIONS)
D = np.zeros((K, K))
G = {k: np.zeros((K, K)) for k in FEAT_KEYS}
with torch.no_grad():
    for r in range(R):
        mixes = [mixture(f) for f in FRACTIONS]     # fresh draws each repeat
        for i in range(K):
            for j in range(K):
                D[i, j] += float(aligner.forward(mixes[i][0], mixes[i][1], mixes[j][0], mixes[j][1]))
                for k in FEAT_KEYS:
                    G[k][i, j] += aligner.last_feature_losses.get(k, 0.0)
D /= R
for k in FEAT_KEYS:
    G[k] /= R
np.set_printoptions(precision=4, suppress=True)
print(f"total energy-distance grid (mean of R={R}, N_MIX={N_MIX}):\n", D)

In [ ]:
import matplotlib.pyplot as plt
labs = [f"{f:.0%}" for f in FRACTIONS]
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(D, origin="lower", cmap="viridis")
ax.set_xticks(range(K)); ax.set_yticks(range(K))
ax.set_xticklabels(labs); ax.set_yticklabels(labs)
ax.set_xlabel("mixture B (% crossing)"); ax.set_ylabel("mixture A (% crossing)")
ax.set_title(f"slice-aligner energy distance (total)\n(crossing/coherent mixtures, t=7\u21928, mean of {R})")
for i in range(K):
    for j in range(K):
        ax.text(j, i, f"{D[i,j]:.3f}", ha="center", va="center",
                color="white" if D[i, j] < D.max() * 0.6 else "black", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="energy distance")
fig.tight_layout()

### Per-feature grids: marginal only / path only / common-ancestor only

Same mixtures, each feature's raw (unweighted) energy distance on its own. The marginal
(`time`) is ~flat (both regimes share the leaf marginal); the lineage features, path
(`slice`) and especially `common_ancestor`, carry the crossing-vs-coherent signal. On a
shared scale the marginal (~100x smaller) reads flat.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from synthetic_evals import viz
viz.apply_rc()

# "How the aligner sees crossing": marginal vs common-ancestor, the shared
# energy-distance colormap (viz.SEQ_CMAP) on a single 0->max scale so the
# marginal -- which carries no crossing/coherent signal -- reads flat.
PANEL = ["time", "common_ancestor"]
NAMES = {"time": "marginal", "common_ancestor": "common ancestor"}
labs = [f"{f:.0%}" for f in FRACTIONS]
vmax = max(G[k].max() for k in PANEL)
norm = Normalize(0.0, vmax)
cmap = viz.SEQ_CMAP

fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.0), constrained_layout=True)
for ax, k in zip(axes, PANEL):
    im = ax.imshow(G[k], origin="lower", cmap=cmap, norm=norm, aspect="equal")
    ax.set_xticks(range(K)); ax.set_xticklabels(labs)
    ax.set_yticks(range(K)); ax.set_yticklabels(labs if ax is axes[0] else [])
    ax.set_xlabel("crossing fraction (B)")
    if ax is axes[0]:
        ax.set_ylabel("crossing fraction (A)")
    ax.set_title(NAMES[k], fontsize=viz.FONT_TITLE)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.6); s.set_color("0.75")
cb = fig.colorbar(im, ax=axes, fraction=0.045, pad=0.02)
cb.set_label("energy distance", fontsize=viz.FONT_LABEL)
cb.outline.set_linewidth(0.6)
cb.ax.tick_params(labelsize=viz.FONT_BASE - 1.0, length=2)
fig.suptitle("how the slice aligner sees crossing vs coherent",
             fontsize=viz.FONT_TITLE + 0.5, y=1.03)

# Production export (vector PDF + SVG + preview PNG).
figdir = ROOT / "figures" / "aligner_mixture_grid"
figdir.mkdir(parents=True, exist_ok=True)
for ext in ("pdf", "svg", "png"):
    fig.savefig(figdir / f"aligner_mixture_grid.{ext}", bbox_inches="tight",
                dpi=300 if ext == "png" else None)
print("wrote", figdir / "aligner_mixture_grid.{pdf,svg,png}")

In [ ]:
# Scatter: mixture distance |f_A - f_B| vs per-feature energy distance (all pairs).
import numpy as np
import matplotlib.pyplot as plt
from synthetic_evals import viz
viz.apply_rc()

fracs = np.asarray(FRACTIONS)
SERIES = [("time", "marginal", viz.PRESPLIT, -0.012),
          ("slice", "path", viz.GS, 0.0),
          ("common_ancestor", "common ancestor", viz.MODEL, 0.012)]

fig, ax = plt.subplots(figsize=(4.4, 3.2), constrained_layout=True)
for k, name, col, dx in SERIES:
    xs = np.array([abs(fracs[i] - fracs[j]) for i in range(K) for j in range(K)])
    ys = np.array([G[k][i, j] for i in range(K) for j in range(K)])
    ax.scatter(xs + dx, ys, s=12, color=col, alpha=0.5, edgecolor="none", zorder=2)
    ux = np.unique(xs)
    mean = np.array([ys[xs == u].mean() for u in ux])
    ax.plot(ux, mean, color=col, lw=1.4, marker="o", ms=3.5, label=name, zorder=3)

ax.set_xlabel(r"mixture distance  $|f_A - f_B|$")
ax.set_ylabel("energy distance")
ax.set_title("aligner energy distance vs mixture distance")
ax.legend(title="feature", loc="upper left", handlelength=1.2, title_fontsize=viz.FONT_LEGEND)
ax.margins(x=0.04)